# LangGraph for the Reservation Analytics AI Agent

Understanding the exact concepts used by the `ReservationAgent` and to know which advanced features can be learned later.

![Project flow](images/00_project_flow.png)

## Learning path

1. State
2. Nodes
3. Edges
4. Conditional edges
5. `StateGraph`, `compile()`, and `invoke()`
6. Mapping the concepts to `ReservationAgent`
7. Optional concepts: tools, checkpointing, human-in-the-loop, streaming, persistence

Core mental model: **LangGraph is a stateful orchestration layer.**

It coordinates components; it does not replace your extractor, RAG system, resolver, or analytics service.

## Chapter 1 — State: the shared working context

![LangGraph State](images/01_state.png)

A **state** is the shared data object that moves through the graph. Each node reads fields from the current state and returns updates.

In the project, `AgentState` may contain values such as:

```python
{
    "question": "How many reservations did campaign X get?",
    "intent": "analytics",
    "metric": "reservation_count",
    "query": {...},
    "resolved_context": {...},
    "status": "answered",
    "answer": "12,345"
}
```

Why this matters: without a shared state, every function would need a growing list of parameters. With LangGraph, the contract becomes approximately `node(state) -> state update`.

In [1]:
from typing import TypedDict

class DemoState(TypedDict, total=False):
    question: str
    intent: str
    status: str
    answer: str

state: DemoState = {"question": "How many reservations?"}
state

{'question': 'How many reservations?'}

### Project connection

Your `extract_node()` adds structured fields to the state:

```python
return {
    **state,
    "extracted": request.model_dump(),
    "intent": request.intent,
    "metric": request.metric,
    "detail_requested": request.detail_requested,
    "query": request.query.model_dump(),
}
```

## Chapter 2 — Nodes: one processing step

![LangGraph Nodes](images/02_nodes.png)

A **node** is a callable that performs one step of work. It can call plain Python logic, an LLM, a retriever, a database service, or another application component.

In this project:

- `extract_node` → calls `RequestExtractor`
- `knowledge_node` → calls `KnowledgeRAG`
- `validate_node` → checks required business context
- `resolve_node` → calls `CampaignResolver`
- `analytics_node` → calls `AnalyticsService`

LangGraph itself does not perform these business operations. It decides **when** they run.

In [2]:
def simple_extract_node(state: DemoState) -> DemoState:
    question = state["question"].lower()
    intent = "analytics" if "how many" in question else "knowledge"
    return {**state, "intent": intent}

simple_extract_node({"question": "How many reservations?"})

{'question': 'How many reservations?', 'intent': 'analytics'}

## Chapter 3 — Edges: deterministic transitions

![LangGraph Edges](images/03_edges.png)

An **edge** connects two nodes. A normal edge means that after one node finishes, execution continues to a specific next node.

Examples from the project:

```python
graph.add_edge(START, "extract")
graph.add_edge("knowledge", END)
graph.add_edge("analytics", END)
```

Read them as:

```text
START -> extract
knowledge -> END
analytics -> END
```

`START` and `END` are special graph markers, not business nodes.

## Chapter 4 — Conditional edges: runtime routing

![LangGraph Conditional Edges](images/04_conditional_edges.png)

A **conditional edge** chooses the next node from the current state.

This is the most important LangGraph concept in your project.

```python
graph.add_conditional_edges(
    "extract",
    self._after_extract,
    {
        "knowledge": "knowledge",
        "validate": "validate",
    },
)
```

Interpretation:

1. Finish the `extract` node.
2. Call `_after_extract(state)`.
3. If it returns `"knowledge"`, go to the `knowledge` node.
4. If it returns `"validate"`, go to the `validate` node.

In [3]:
def after_extract(state: DemoState) -> str:
    return "knowledge" if state["intent"] == "knowledge" else "validate"

print(after_extract({"intent": "knowledge"}))
print(after_extract({"intent": "analytics"}))

knowledge
validate


## Chapter 5 — Build, compile, and invoke a graph

![LangGraph concept summary](images/10_concept_summary.png)

The minimum lifecycle is:

```text
State schema
   -> StateGraph
   -> add nodes
   -> add edges / conditional edges
   -> compile()
   -> invoke(initial_state)
```

`compile()` turns the graph definition into an executable graph. `invoke()` runs it from an initial state until it reaches `END`.

In [5]:
# Run after: pip install langgraph
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class MiniState(TypedDict, total=False):
    question: str
    intent: str
    answer: str


def extract(state: MiniState):
    return {
        "intent": "analytics" if "how many" in state["question"].lower() else "knowledge"
    }


def knowledge(state: MiniState):
    return {"answer": "A reservation is ..."}


def analytics(state: MiniState):
    return {"answer": "Reservation count = 1,234"}


def route(state: MiniState):
    return state["intent"]

builder = StateGraph(MiniState)
builder.add_node("extract", extract)
builder.add_node("knowledge", knowledge)
builder.add_node("analytics", analytics)
builder.add_edge(START, "extract")
builder.add_conditional_edges(
    "extract", route,
    {"knowledge": "knowledge", "analytics": "analytics"},
)
builder.add_edge("knowledge", END)
builder.add_edge("analytics", END)

graph = builder.compile()
# graph.invoke({"question": "How many reservations?"})

## Chapter 6 — Map LangGraph directly to `ReservationAgent`

![Project flow](images/00_project_flow.png)

The project has two main paths.

### Path A: knowledge question

```text
START
  -> extract
  -> knowledge
  -> END
```

Example:

```text
What does reservation_count mean?
```

This goes to RAG because the user is asking for a definition, not a business number.

### Path B: analytics question

```text
START
  -> extract
  -> validate
  -> resolve
  -> analytics
  -> END
```

Example:

```text
How many reservations did the M Company 17 Pro Germany campaign get?
```

This path deliberately validates business context and resolves the campaign before calling analytics.

### 6.1 `extract_node`: natural language -> structured request

The extractor produces fields such as `intent`, `metric`, `detail_requested`, and `query`.

Important architecture point:

> The LLM/extractor interprets language, but the graph controls the workflow.

This keeps language understanding separate from data execution.

### 6.2 `validate_node`: do not guess missing business context

The project checks:

```python
missing = missing_context(query)
```

If required context is missing, the graph returns a clarification rather than inventing filters.

A strong project discussion explanation is:

> We explicitly validate required business context before analytics execution so the agent does not guess campaign, market, or other critical filters.

### 6.3 `resolve_node`: business entity disambiguation

The resolver maps a natural-language business description to a specific campaign.

```text
0 matches    -> not_found / stop
1 match      -> resolved / continue
>1 matches   -> clarification / stop
```

This is important because the analytics service should receive a trusted campaign identity rather than an ambiguous phrase from the user.

### 6.4 `analytics_node`: execute only after context is trusted

Only after validation and resolution does the graph call:

```python
self.analytics.run(
    state["metric"],
    campaign,
    state.get("detail_requested", False),
)
```

The architecture is therefore:

```text
Natural language
    -> Structured request
    -> Validation
    -> Entity resolution
    -> Trusted analytics service
    -> Answer
```

This is more controlled than sending every question directly to an LLM-generated arbitrary SQL query.

## Chapter 7 — Tool integration: useful next, but not the core of this project

![LangGraph Tools Integration](images/05_tools_integration.png)

LangGraph can orchestrate tool calls such as search, APIs, or database functions. The project already follows a similar architectural principle, but `KnowledgeRAG`, `CampaignResolver`, and `AnalyticsService` are called directly from nodes rather than being exposed as generic LLM tools.

For the current project, learn this concept at a high level only. Do not make it your first priority.

## Chapter 8 — Memory and checkpointing: optional for later

![LangGraph Memory and Checkpointing](images/06_memory_checkpointing.png)

Checkpointing stores graph state so a workflow can be resumed or inspected later. This becomes useful for multi-turn conversations, long-running workflows, retry/recovery, and human approval steps.

Your current `ReservationAgent` does not need deep checkpointing knowledge to understand its core flow.

## Chapter 9 — Human-in-the-loop: clarification is conceptually related

![LangGraph Human in the Loop](images/07_human_in_the_loop.png)

Human-in-the-loop workflows pause execution and wait for user or operator input before continuing.

Your current project already has a lightweight business version of this idea: it stops when context is missing or when multiple campaigns match, and asks the user to clarify. A more advanced implementation could persist the graph and resume it after the clarification arrives.

## Chapter 10 — Streaming: useful for UX, not required for understanding the graph

![LangGraph Streaming](images/08_streaming.png)

Streaming can expose intermediate node events or partial model output while the graph runs. It improves UI responsiveness, but it does not change the core concepts of state, nodes, edges, and routing.

Treat this as a later enhancement for a chat UI or service endpoint.

## Chapter 11 — Persistence and durable execution: advanced production capability

![LangGraph Persistence](images/09_persistence.png)

Durable execution makes a workflow recoverable across process restarts or long pauses. It is valuable for long-running agent workflows, approvals, and reliable production execution.

For the reservation analytics project, this is advanced material. The current graph is short-lived and can be understood without it.

## Chapter 12 — Why the manual fallback is a good design clue

The project contains a fallback path when LangGraph is unavailable:

```python
if self._graph is not None:
    return self._graph.invoke(state)

state = self.extract_node(state)
...
return self.analytics_node(state)
```

This reveals an important design principle:

> **Business logic is decoupled from orchestration.**

The nodes are ordinary Python methods. LangGraph coordinates them, but the business operations remain independently testable.

## Chapter 13 — Presentation-ready explanation

A concise explanation:

> We use LangGraph as the orchestration layer for the reservation analytics agent. The graph first extracts a structured request. Knowledge questions are routed to RAG, while analytics questions pass through business-context validation and campaign resolution before reaching the analytics service. Conditional edges make routing, clarification, and stopping conditions explicit, so the LLM does not implicitly control every decision.

### Key questions

| Question | Short answer |
|---|---|
| What is LangGraph? | A stateful orchestration framework for graph-based agent workflows. |
| What is State? | Shared context read and updated across nodes. |
| What is a Node? | A callable that performs one processing step. |
| What is an Edge? | A transition from one node to another. |
| What is a Conditional Edge? | A runtime route chosen from the current state. |
| Why `compile()`? | It turns the graph definition into an executable graph. |
| Why `invoke()`? | It executes the graph from an initial state. |
| Why use it here? | To make routing, validation, clarification, and stopping logic explicit. |

## Chapter 14 — What to learn now vs. later

### Learn now

```text
State
 -> Node
 -> Edge
 -> Conditional Edge
 -> START / END
 -> StateGraph
 -> compile()
 -> invoke()
```

### Learn later

- Tool nodes and generic tool calling
- Checkpointers
- Human-in-the-loop APIs
- Streaming
- Durable execution / persistence
- Subgraphs
- Parallel branches
- Multi-agent patterns

If you fully understand the first list, you can already explain most of the `ReservationAgent` implementation.

## Chapter 15 — Practice exercise

Build a small graph with this behavior:

```text
question
   -> extract
   -> knowledge? -- yes --> answer -> END
          |
          no
          v
       validate
          |
       analytics
          |
         END
```

Start from the skeleton below.

In [ ]:
from typing import TypedDict

class ReservationState(TypedDict, total=False):
    question: str
    intent: str
    status: str
    answer: str


def extract(state: ReservationState):
    # "what is" -> knowledge
    # otherwise -> analytics
    ...


def knowledge(state: ReservationState):
    return {
        "status": "answered",
        "answer": "reservation_count means ...",
    }


def validate(state: ReservationState):
    ...


def analytics(state: ReservationState):
    return {
        "status": "answered",
        "answer": "Reservation count = 1,234",
    }

# Build the StateGraph here.